# 9. KLF13 repression targets: expression shifts in GABAergic cells

To summarize how Klf13 knockout affects its predicted repression targets, we combine prior-knowledge TF–target networks with our GABAergic single-cell data. In this notebook, we:

- extract KLF13 targets from **DoRothEA** and **CollecTRI** (mouse),
- select predicted **repression targets** (negative interaction weights),
- compute KO − WT expression shifts for these targets within the GABAergic subset, and
- visualize the shifts in a publication-style lollipop plot.

Each point in the plot corresponds to one KLF13 repression target; the x-axis shows the KO − WT expression difference (log-normalized) in GABAergic cells.

## 9.1. Imports and settings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import scanpy as sc
import scipy.sparse as sp
import dorothea
import decoupler as dc

# Scanpy settings
sc.settings.verbosity = 0
sc.settings.set_figure_params(dpi=80, facecolor="white", frameon=False)

# Reproducibility
SEED = 123
np.random.seed(SEED)

## 9.2. Load KLF13 repression targets from DoRothEA and CollecTRI

We first extract KLF13 targets from:
- DoRothEA mouse regulons (levels A–C), and
- CollecTRI (mouse).

We then combine these sources and retain only targets with **negative weights**, corresponding to predicted repression targets.

In [ ]:
# Load DoRothEA regulons (wide matrix: genes × TFs)
doro_wide = dorothea.load_regulons(
    organism="Mouse",
    levels=["A", "B", "C"],
    commercial=False,
)

# Extract Klf13 column: non-zero entries correspond to targets
klf13_col = doro_wide["Klf13"]
doro_df = pd.DataFrame(
    {
        "target": klf13_col[klf13_col != 0].index,
        "weight": klf13_col[klf13_col != 0].values,
    }
)

# Try to load CollecTRI KLF13 targets
try:
    ct_net = dc.get_collectri(organism="mouse", split_complexes=False)
    klf13_ct = ct_net.loc[ct_net["source"] == "Klf13", ["target", "weight"]]
except Exception:
    klf13_ct = pd.DataFrame(columns=["target", "weight"])

# Combine DoRothEA and CollecTRI
combined = (
    pd.concat([doro_df, klf13_ct])
    .drop_duplicates(subset="target", keep="first")
    .reset_index(drop=True)
)

# Keep repression targets (weight < 0)
rep_genes = combined.loc[combined["weight"] < 0, "target"].tolist()
print(f"Number of KLF13 repression targets: {len(rep_genes)}")
print(rep_genes)

## 9.3. Load GABAergic AnnData and prepare expression

We now load the GABA-reclustered AnnData object and apply basic filtering to:

- remove any unwanted clusters (e.g. a small outlier cluster),
- restrict to samples with sufficient cell counts, and
- choose the expression layer used for expression shift calculations.

We then compute KO − WT expression shifts for each KLF13 repression target.

In [ ]:
cluster_key   = "leiden_0.7"
condition_key = "condition"
sample_key    = "batch"
ref_group     = "WT"
test_group    = "KO"

adata = sc.read_h5ad("gaba_reclust_scran0_7.h5ad")

# Clean and standardize metadata
adata.obs[cluster_key]   = adata.obs[cluster_key].astype(str)
adata.obs[condition_key] = adata.obs[condition_key].astype(str).str.strip().str.upper()
adata.obs[sample_key]    = adata.obs[sample_key].astype(str)

# Optional: remove a cluster not used downstream
if "14" in adata.obs[cluster_key].unique():
    adata = adata[adata.obs[cluster_key] != "14"].copy()

# Optional: keep only samples with sufficient cells and drop a problematic sample
sc_counts = adata.obs[sample_key].value_counts()
keep_samples = [s for s in sc_counts[sc_counts >= 200].index if s != "96_6_M_KO"]
adata = adata[adata.obs[sample_key].isin(keep_samples)].copy()
print(f"Cells after sample filtering: {adata.n_obs}")

# Use log1p-normalized counts if available
if "log1p_norm" in adata.layers:
    adata.X = adata.layers["log1p_norm"].copy()

# Convert to dense and clean NaN/inf
X = adata.X.toarray() if sp.issparse(adata.X) else np.array(adata.X)
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

expr_df = pd.DataFrame(X, index=adata.obs_names, columns=adata.var_names)

# Masks for WT and KO cells
wt_mask = (adata.obs[condition_key] == ref_group.upper()).values
ko_mask = (adata.obs[condition_key] == test_group.upper()).values

# Compute KO − WT mean expression shift for each repression target
fc_map = {}
for gene in rep_genes:
    if gene in expr_df.columns:
        fc_map[gene] = expr_df.loc[ko_mask, gene].mean() - expr_df.loc[wt_mask, gene].mean()
    else:
        fc_map[gene] = 0.0

# Build a DataFrame sorted by expression shift
df = pd.DataFrame(
    {
        "gene": rep_genes,
        "fc":   [fc_map[g] for g in rep_genes],
    }
).sort_values("fc", ascending=True).reset_index(drop=True)

df.head()

## 9.4. Lollipop plot of KLF13 repression targets

We now generate a lollipop plot where:

- each horizontal line and dot corresponds to one KLF13 repression target,
- the x-axis shows KO − WT expression (log-normalized units),
- red points indicate targets upregulated in KO (de-repressed),
- blue points indicate targets downregulated in KO (consistent with repression),
- grey points show minimal change.

This figure can be exported as a high-resolution PNG for use in the thesis.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 10), facecolor="white")

y_pos = np.arange(len(df))
fc_val = df["fc"].values

# Color scheme: up in KO, down in KO, no change
colors = []
for v in fc_val:
    if v > 0.005:
        colors.append("#C0392B")  # red: up in KO (de-repressed)
    elif v < -0.005:
        colors.append("#2980B9")  # blue: down in KO
    else:
        colors.append("#AAAAAA")  # grey: no change

# Horizontal stems from x = 0
ax.hlines(y_pos, 0, fc_val, colors="#DDDDDD", linewidth=1.5, zorder=1)

# Dots at ends of stems
ax.scatter(
    fc_val,
    y_pos,
    c=colors,
    s=120,
    zorder=3,
    edgecolors="white",
    linewidths=0.8,
)

# Gene labels
for i, (_, row) in enumerate(df.iterrows()):
    x = row["fc"]
    x_offset = 0.0008 if x >= 0 else -0.0008
    ha = "left" if x >= 0 else "right"
    ax.text(
        x + x_offset,
        i,
        row["gene"],
        va="center",
        ha=ha,
        fontsize=8.5,
        fontweight="bold",
        color="#333333",
    )

# Zero line
ax.axvline(0, color="black", linewidth=1.0, linestyle="-", zorder=2)

# Axis labels and title
ax.set_yticks([])
ax.set_xlabel("Expression shift in KO  (KO − WT, log-normalised)", fontsize=10)
ax.set_title(
    "KLF13 repression targets\nDoRothEA + CollecTRI (Mouse, levels A–C)",
    fontsize=12,
    fontweight="bold",
    pad=15,
)

# Legend
ax.legend(
    handles=[
        mpatches.Patch(color="#C0392B", label="Up in KO (de-repressed)"),
        mpatches.Patch(color="#2980B9", label="Down in KO"),
        mpatches.Patch(color="#AAAAAA", label="No change"),
    ],
    frameon=False,
    fontsize=9,
    loc="lower right",
)

# Clean aesthetics
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(axis="x", labelsize=9)

# Ensure labels are not clipped
xl = ax.get_xlim()
span = abs(xl[1] - xl[0])
ax.set_xlim(xl[0] - span * 0.35, xl[1] + span * 0.35)

plt.tight_layout()
plt.savefig("klf13_lollipop.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved lollipop figure as 'klf13_lollipop.png'")

## 9.5. Numeric summary of KO − WT shifts (optional)

For reference, we print the numerical KO − WT expression shift for each repression target.

In [ ]:
summary_df = df[["gene", "fc"]].rename(columns={"fc": "KO-WT shift"})
print(summary_df.to_string(index=False))